# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list the available record sets, their `@id`s, and the fields they contain, also referenced by their field and column `@id`s. This will help in referencing them precisely in the following sections.

In [ ]:
# List the available record sets and corresponding fields with `@id`s
from pprint import pprint
record_sets = metadata.record_set
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    if hasattr(rs, 'field'):
        print("  Fields:")
        for field in rs.field:
            field_id = field['@id'] if isinstance(field, dict) else getattr(field, '@id', str(field))
            print(f"    - {field_id}")
    if hasattr(rs, 'column'):
        print("  Columns:")
        for column in rs.column:
            column_id = column['@id'] if isinstance(column, dict) else getattr(column, '@id', str(column))
            print(f"    - {column_id}")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below we create a list of all record set `@id`s available in the dataset, and extract their records as pandas DataFrames for further analysis. You can update `selected_record_set_id` to the specific record set you want to work with.

In [ ]:
# Build a list of available record set `@id`s
record_sets = metadata.record_set
record_set_ids = [rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', str(rs)) for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if record_set_ids:
    selected_record_set_id = record_set_ids[0]  # you can change this index to another record set
    print(f"Fields/Columns in record set {selected_record_set_id}:")
    print(dataframes[selected_record_set_id].columns.tolist())
    dataframes[selected_record_set_id].head()
else:
    print("No record sets available to extract.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Update `numeric_field_id` and `group_field_id` below with the appropriate `@id` from your data columns (listed above).

In [ ]:
# Set up analysis variables for field IDs from your actual data columns
record_set_id = selected_record_set_id

# For demonstration, select a numeric field from the DataFrame columns
# If analyzing your own data, set these based on step 3 output
df = dataframes[record_set_id]

numeric_field_candidates = df.select_dtypes(include='number').columns.tolist()
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]  # update this for analysis of another field
else:
    print("No numeric fields found. Please check your DataFrame.")
    numeric_field_id = None

threshold = 10
if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Pick a group field
    candidate_group_fields = df.select_dtypes(include='object').columns.tolist()
    group_field_id = candidate_group_fields[0] if candidate_group_fields else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (showing top 5 groups):")
        print(grouped_df.head())
    else:
        print("No group field found for grouping.")
else:
    print("No numeric field selected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following example demonstrates plotting a histogram for the numeric field and a boxplot grouped by the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    filtered_df[numeric_field_id].plot(kind='hist', bins=20, title=f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to load a Croissant-structured dataset and inspect its structure using entity `@id`s for reference, as well as how to filter, normalize, and visualize data fields.
- Further, in-depth EDA can be conducted by adjusting field selections based on the columns present in your dataset.
- Explore other record sets and fields by updating variables in the notebook for more targeted analyses.